# Simulator smoke test

Run from `simulator/`. All logic lives in the `.py` files; this notebook only imports and plots.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, so `simulator` imports

import numpy as np
import matplotlib.pyplot as plt
from simulator import simulator_config as cfg
from simulator.environment.dispatch_env import DispatchEnv
from simulator.agent_interface import contract as C

## One closest-unit episode

In [ ]:
env = DispatchEnv()
obs, info = env.reset(seed=0, options={"load": 0.6})
fleet, times, records, done = [], [], [], False
while not done:
    fleet.append(obs["policy"][C.OBS_SLICES["fleet"]]); times.append(env.sim.now / 3600)
    obs, r, done, _, info = env.step(info["closest_road"])   # closest by ETA (road time)
    records += info["calls"]
sim = env.sim
print(f"map={sim.city.name} load={sim.load} start_hour={sim.start_hour}")
print(f"calls={len(sim.calls)} decisions={len(times)} episode end={sim.now/3600:.2f} h")
for t, kind, unit, call in sim.log[:25]:
    print(f"{t:8.0f}s  {kind:<18} unit={unit}  call={call}")

## Response times per severity

In [ ]:
T = np.array([r["T"] for r in records]); sev = np.array([r["severity"] for r in records])
fig, ax = plt.subplots(figsize=(7, 3))
for s, name in enumerate(cfg.SEVERITIES):
    ax.hist(T[sev == s], bins=30, alpha=0.6, label=f"{name} (on-time {np.mean(T[sev == s] <= cfg.REWARD_TAU[s]):.0%})")
ax.axvline(300, color="k", ls="--", lw=1); ax.set_xlabel("response time T (s)"); ax.legend()
plt.show()

## Fleet state at each decision

In [ ]:
F = np.array(fleet)
fig, ax = plt.subplots(figsize=(7, 3))
ax.stackplot(times, F.T, labels=cfg.UNIT_STATES, step="post")
ax.set_xlabel("sim hours"); ax.set_ylabel("share of fleet"); ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))
plt.show()